In [24]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.data.dataloader import MIDASDataset
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torchvision.models import resnet50, ResNet50_Weights

In [22]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
metadata = pd.read_csv("/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/midas.csv")

train_ids, test_ids = train_test_split(metadata["midas_record_id"].unique(), test_size=0.15, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.1765, random_state=42)

train = metadata[metadata["midas_record_id"].isin(train_ids)]
val = metadata[metadata["midas_record_id"].isin(val_ids)]
test = metadata[metadata["midas_record_id"].isin(test_ids)]
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

train_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="train.csv", transform=transform)
val_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="val.csv", transform=transform)
test_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="test.csv", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [29]:
class MelanomaDetection(nn.module):
    def __init__(self,):
        super(MelanomaDetection, self).__init()

NameError: name 'nn' is not defined

In [35]:
import os
import random
import torch
import torch.nn as nn
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

IMAGE_TYPE_MAP = {"dscope": 0, "6in": 1, "1ft": 2, "n/a - virtual": 3}

class MIDASModel(nn.Module):

    def __init__(self):

        super().__init__()

        # image encoder
        self.cnn = models.resnet18(pretrained=True)

        # remove final classifier
        self.cnn.fc = nn.Identity()

        # image type embeddings
        self.type_embedding = nn.Embedding(4, 32)

        # classifier
        self.classifier = nn.Sequential(

            nn.Linear(512 + 32, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, 1)
        )


    def forward(self, images, image_types):

        """
        images:
        [N, 3, 224, 224]

        image_types:
        [N]
        """

        # image features
        image_features = self.cnn(images)

        # image type embeddings
        type_features = self.type_embedding(image_types)

        # combine
        combined = torch.cat(
            [image_features, type_features],
            dim=1
        )

        # aggregate patient images
        patient_feature = combined.mean(dim=0)

        # classify
        output = self.classifier(patient_feature)

        return output


# =========================================================
# INITIALIZE MODEL
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MIDASModel().to(device)


# =========================================================
# LOSS + OPTIMIZER
# =========================================================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


# =========================================================
# TRAINING LOOP
# =========================================================

num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for images, image_types, label in train_loader:

        # remove batch dimension
        images = images[0].to(device)

        image_types = image_types[0].to(device)

        label = label.to(device)

        optimizer.zero_grad()

        output = model(images, image_types)

        loss = criterion(
            output.unsqueeze(0),
            label.unsqueeze(0)
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f}"
    )


# =========================================================
# SIMPLE EVALUATION
# =========================================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, image_types, label in test_loader:

        images = images[0].to(device)

        image_types = image_types[0].to(device)

        label = label.to(device)

        output = model(images, image_types)

        prediction = torch.sigmoid(output)

        prediction = (prediction > 0.5).float()

        correct += (prediction == label).sum().item()

        total += 1

print(f"Test Accuracy: {correct / total:.4f}")

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 | Loss: 0.4472
Epoch 2 | Loss: 0.4336
Epoch 3 | Loss: 0.4421
Epoch 4 | Loss: 0.4197
Epoch 5 | Loss: 0.4303
Test Accuracy: 0.8350
